In [118]:
import numpy as np

types_list = [
    "wuetend",       # 0
    "gluecklich",    # 1
    "gelangweilt",  # 2
    "ueberrascht",   # 3
    "aengstlich",    # 4
    "angeekelt",    # 5
    "traurig",      # 6
]

adj = np.array([
    # wü  gl  ge  üb  äg  an  tr
    [  0,  0,  1,  0,  1,  0,  1],  # 0 wütend      → ge, äg, tr
    [  1,  0,  0,  0,  1,  1,  0],  # 1 glücklich   → wü, äg, an
    [  0,  1,  0,  0,  0,  1,  1],  # 2 gelangweilt → gl, an, tr
    [  1,  1,  0,  0,  1,  0,  0],  # 3 überrascht  → wü, gl, äg
    [  0,  0,  1,  0,  0,  1,  1],  # 4 ängstlich   → ge, an, tr
    [  1,  0,  0,  1,  0,  0,  1],  # 5 angeekelt   → wü, üb, tr
    [  0,  1,  0,  1,  1,  0,  0],  # 6 traurig     → gl, üb, äg
])

In [250]:
import json
from pathlib import Path
from PIL import Image, ImageChops, ImageDraw, ImageFont

cards_path = Path("cards.json")

with cards_path.open("r", encoding="utf-8") as f:
    cards = json.load(f)

assets_dir = Path("assets")
export_dir = Path("exports")
final_dir = Path("final")
fonts_dir = Path("fonts")
icons_dir = Path("icons")

lato_title_index = ImageFont.truetype(str(fonts_dir / "Lato" / "Lato-Light.ttf"), 120)
lato_title_name = ImageFont.truetype(str(fonts_dir / "Lato" / "Lato-Bold.ttf"), 120)
lato_text = ImageFont.truetype(str(fonts_dir / "Lato" / "Lato-Regular.ttf"), 100)
lato_small_text = ImageFont.truetype(str(fonts_dir / "Lato" / "Lato-Regular.ttf"), 80)

for idx, card in enumerate(cards):
    creature_number = card["index"]
    creature_name = card["name"]
    card_file_name = f"{creature_number:03d}_{creature_name}.png"
    types = card["type"]
    first_type = types[0]
    back_path = assets_dir / f"back_{first_type}_{len(types)}.png"
    img_path = export_dir / card_file_name
    final_path = final_dir / card_file_name

    print(creature_number, creature_name)
    # print(img_path)

    back_img = Image.open(back_path).convert("RGBA")
    if img_path.is_file():
        front_img = Image.open(img_path).convert("RGBA")
        if "imageMultiply" in card and card["imageMultiply"] is True:
            result = ImageChops.multiply(back_img, front_img)
        else:
            result = back_img.copy()
            result.alpha_composite(front_img)
    else:
        result = back_img.copy()

    draw = ImageDraw.Draw(result)
    draw.text((60, 35), f"{creature_number:03d}", font=lato_title_index, fill=(0, 0, 0, 255))
    draw.text((280, 35), f"{creature_name}", font=lato_title_name, fill=(0, 0, 0, 255))

    is_neutral = False
    if first_type not in types_list:
        is_neutral = True

    icon_size = 500
    types.reverse()
    for idx, type in enumerate(types):
        type_img_path = assets_dir / f"{type}.png"
        type_img = Image.open(type_img_path).convert("RGBA")
        type_img = type_img.resize((icon_size, icon_size), Image.Resampling.LANCZOS)
        if is_neutral:
            result.alpha_composite(type_img, (result.size[0]-480, -75))
        else:
            result.alpha_composite(type_img, (result.size[0]-445 - 225*idx, -90))



    attacks = card["attacks"]
    attack_icon_size = 260
    small_icon_size = 250

    defense_img_path = assets_dir / "verteidigung.png"
    defense_img = Image.open(defense_img_path).convert("RGBA")
    defense_img = defense_img.resize((280, 280), Image.Resampling.LANCZOS)
    result.alpha_composite(defense_img, (result.size[0]-300, 1160))
    draw.text((result.size[0]-300+105, 1160+40), f"{card['defense']}", font=lato_title_name, fill=(0, 0, 0, 255))


    for idx, attack in enumerate(attacks):
        y_pos = 1565 + idx*225
        if idx == 1:
            attack_img_path = assets_dir / "lade_attacke.png"
        else:
            attack_img_path = assets_dir / "standard_attacke.png"

        attack_img = Image.open(attack_img_path).convert("RGBA")
        attack_img = attack_img.resize((attack_icon_size, attack_icon_size), Image.Resampling.LANCZOS)
        result.alpha_composite(attack_img, (35, y_pos))

        attack_type_img_path = assets_dir / f"{attack['type']}.png"
        attack_type_img = Image.open(attack_type_img_path).convert("RGBA")
        attack_type_img = attack_type_img.resize((small_icon_size, small_icon_size), Image.Resampling.LANCZOS)
        result.alpha_composite(attack_type_img, (35 + 70, y_pos + 10))

        draw.text((310, y_pos+55), f"{attack['damage']}", font=lato_title_name, fill=(0, 0, 0, 255))
        draw.text((400, y_pos+75), f"{attack['name']}", font=lato_text, fill=(0, 0, 0, 255))

        for i in range(attack['energy']):
            energy_img_path = assets_dir / "energie.png"
            energy_img = Image.open(energy_img_path).convert("RGBA")
            energy_img = energy_img.resize((small_icon_size, small_icon_size), Image.Resampling.LANCZOS)
            if is_neutral:
                result.alpha_composite(energy_img, (result.size[0]-400 - 50*i, y_pos + 10))
            else:
                result.alpha_composite(energy_img, (result.size[0]-630 - 50*i, y_pos + 10))

        draw.text((result.size[0]-200, y_pos+55), f"+{attack['damage']}", font=lato_title_name, fill=(0, 0, 0, 255))

        print(f"{attack['name']} ({attack['type']})")
        if is_neutral:
            # print(f"neutral type: {attack['type']}")
            continue
        type_idx = types_list.index(attack['type'])
        # print(attack['type'], type_idx, adj[type_idx, :])
        advantages = np.argwhere(adj[type_idx, :] == 1).flatten()
        advantages = np.flip(advantages)
        
        for i, adv_idx in enumerate(advantages):
            adv_type = types_list[adv_idx]
            # print(f"  --> {adv_type}")
            adv_img_path = assets_dir / f"{adv_type}.png"
            adv_img = Image.open(adv_img_path).convert("RGBA")
            adv_img = adv_img.resize((small_icon_size, small_icon_size), Image.Resampling.LANCZOS)
            result.alpha_composite(adv_img, (result.size[0]-400 - 60*i, y_pos + 10))

    evo_icon_size = 220
    if "evolution" in card:
        evo_print = ""
        x_pos = 50
        y_pos = 1360
        for idx, evolution in enumerate(card["evolution"]):
            icon_image_path = icons_dir / f"{evolution['index']:03d}_{evolution['name']}_icon.png"
            # print(icon_image_path)
            
            if icon_image_path.is_file():
                # print(icon_image_path)
                icon_img = Image.open(icon_image_path).convert("RGBA")
                target_h = evo_icon_size
                w, h = icon_img.size
                target_w = int(w * (target_h / h))
                icon_img = icon_img.resize((target_w, target_h), Image.Resampling.LANCZOS)
                result.alpha_composite(icon_img, (x_pos, y_pos))
                x_pos += target_w
                draw.text((x_pos+10, y_pos+target_h-120), f"{evolution['index']:03d}", font=lato_small_text, fill=(0, 0, 0, 255))
                
                if idx < len(card["evolution"]) - 1:
                    evo_print += f"{evolution['index']:03d} {evolution['name']} → "
                    arrow_img_path = assets_dir / "pfeil.png"
                    arrow_img = Image.open(arrow_img_path).convert("RGBA")
                    arrow_img = arrow_img.resize((evo_icon_size, evo_icon_size), Image.Resampling.LANCZOS)
                    result.alpha_composite(arrow_img, (x_pos, y_pos-10))
                else:
                    evo_print += f"{evolution['index']:03d} {evolution['name']}"

                x_pos += 220
                    
        print(evo_print)




    result.save(final_path)


1 Kruggler
Grummler (gelangweilt)
Stampfer (wuetend)
2 Robi
Blitz (neutral_ungluecklich)
Elektroshock (neutral_ungluecklich)
002 Robi → 003 Robo
3 Robo
Funken (neutral_gluecklich)
Lichtbogen (neutral_gluecklich)
002 Robi → 003 Robo
4 Robella
Funken (neutral_gluecklich)
Lichtbogen (neutral_gluecklich)
5 Regelwurm
Walzer (gluecklich)
Tanzer (gluecklich)
6 Spinfli
Flügelschlag (angeekelt)
Schlecker (ueberrascht)
7 Schleggach
Schlecker (angeekelt)
Lechzer (gluecklich)
8 Samtange
Beißer (aengstlich)
Stampfer (aengstlich)
